# A graph's history joined with its version file

A history can name the file that holds its version. Each cell runs a real port over a small repository.

## Setup

In [ ]:
from __future__ import annotations

import contextlib
import io
import json
import subprocess
import tempfile
from pathlib import Path
from typing import Any

from fastapi.testclient import TestClient
from maxitor.repo_graph_port.cli import main as port_cli
from maxitor.repo_graph_port.server import create_app

BASE = "/api/v1/history"
FOLDER = Path(tempfile.mkdtemp())


def graph(entities: int, order: int = 1) -> str:
    """A valid graph of ``entities`` nodes, in the given order."""
    nodes = [{"id": f"shop.E{n}", "type": "Entity", "label": f"E{n}", "properties": {}} for n in range(entities)][::order]
    return json.dumps({"schema_version": "1.2", "nodes": nodes, "edges": []}, indent=2) + "\n"


def component(number: str) -> str:
    """A component's ``version.toml``."""
    return f'kind = "component"\nversion = "{number}"\n'


class Repository:
    """A git repository the example commits to."""

    def __init__(self, name: str) -> None:
        """Start it, with the folder ``svc``."""
        self.path = FOLDER / name
        (self.path / "svc").mkdir(parents=True)
        self.git("init", "-q", "-b", "main")

    def git(self, *arguments: str) -> None:
        """Run git in the repository."""
        subprocess.run(["git", "-C", str(self.path), "-c", "user.name=T", "-c", "user.email=t@e", *arguments], check=True, capture_output=True)

    def commit(self, message: str, files: dict[str, str]) -> None:
        """Write ``files`` and commit them."""
        for name, text in files.items():
            (self.path / name).parent.mkdir(parents=True, exist_ok=True)
            (self.path / name).write_text(text, encoding="utf-8")
        self.git("add", "-A")
        self.git("commit", "-q", "-m", message)


def read(repository: Repository, settings: str, graph_file: str = "svc/graph.json") -> list[dict[str, Any]] | dict[str, Any]:
    """The versions answer of a port started over ``repository`` with ``settings``."""
    config = repository.path.parent / f"{repository.path.name}-{abs(hash(settings))}.toml"
    config.write_text(
        f'[[history]]\nid = "svc"\nhost = "local"\nrepository = "{repository.path}"\nbranch = "main"\ngraph_file = "{graph_file}"\n{settings}',
        encoding="utf-8",
    )
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        port_cli(["--config", str(config), "key", "new", "--history", "svc", "--name", "anna"])
    client = TestClient(create_app(config, config.with_suffix(".cache")))
    answer: dict[str, Any] = client.get(f"{BASE}/svc/versions", headers={"X-Maxitor-History-Key": out.getvalue().strip()}).json()
    return answer


def show(answer: Any) -> None:
    """One line per point."""
    for point in answer["versions"]:
        graph_word = "graph changed" if point["graph_changed"] else "same graph   "
        version = f'{point["version"] or "—":6} {"new" if point["version_changed"] else "   "}'
        print(f'   {point["message"].strip().ljust(16)} {graph_word}  {version}')

## The points of a history with a version file

Each point says whether the graph changed, which version is there, and whether it changed. A comment in the version file and a reordering of nodes make no point.

In [ ]:
print("1. The points of a history with a version file")
service = Repository("service")
service.commit("first", {"svc/graph.json": graph(2), "svc/version.toml": component("1.0.0")})
service.commit("model and number", {"svc/graph.json": graph(3), "svc/version.toml": component("1.1.0")})
service.commit("code, number", {"svc/code.txt": "x", "svc/version.toml": component("1.1.1")})
service.commit("model only", {"svc/graph.json": graph(4)})
service.commit("a comment", {"svc/version.toml": "# a note\n" + component("1.1.1")})
service.commit("nodes reordered", {"svc/graph.json": graph(4, -1)})
show(read(service, 'version_file = "svc/version.toml"\n'))

## No version file

The points it always had: every commit of the graph file.

In [ ]:
print("2. No version file: the points it always had")
show(read(service, ""))

## A package's file and a key

`aoa-demo` has no components, so its graph uses the package's number. A dependency makes no point; a new number does.

In [ ]:
print("3. A package's file and a key")
package = Repository("package")
package.commit("first", {"svc/graph.json": graph(2), "pyproject.toml": '[project]\nversion = "3.0.0"\n'})
package.commit("a dependency", {"pyproject.toml": '[project]\nversion = "3.0.0"\ndependencies = ["x"]\n'})
package.commit("raised", {"pyproject.toml": '[project]\nversion = "3.1.0"\n'})
show(read(package, 'version_file = "pyproject.toml"\nversion_key = "project.version"\n'))

## Each bad link

Each is explained in plain words, and the graph still opens.

In [ ]:
print("4. Each bad link")
bad = Repository("bad")
bad.commit("first", {"svc/graph.json": graph(2), "svc/version.toml": 'kind = "package"\ndriver = "python"\n', "svc/pom.xml": "<project/>", "other/version.toml": component("1.0.0")})
for settings in (
    'version_file = "other/version.toml"\n',
    'version_file = "svc/gone.toml"\n',
    'version_file = "svc/pom.xml"\n',
    'version_file = "svc/version.toml"\n',
):
    answer = read(bad, settings)
    print(f'   {answer["version_problem"]} ({len(answer["versions"])} version, the graph opens)')

## A moved folder

Old paths keep the history before the move. Without them the history starts at the move.

In [ ]:
print("5. A moved folder")
moved = Repository("moved")
moved.commit("first", {"svc/graph.json": graph(2), "svc/version.toml": component("1.0.0")})
moved.git("mv", "svc", "new")
moved.commit("moved", {"new/version.toml": component("1.1.0")})
show(read(moved, 'version_file = "new/version.toml"\ngraph_file_before = ["svc/graph.json"]\nversion_file_before = ["svc/version.toml"]\n', "new/graph.json"))
print("   without the old paths:")
show(read(moved, 'version_file = "new/version.toml"\n', "new/graph.json"))